# 🎙️ MeetingMind — Pipeline Bóc Băng Chuẩn Nhất

> **Mục tiêu:** Đạt độ chính xác tối đa với cả **Tiếng Việt** và **Tiếng Anh**

---

## 🧠 Chiến lược được áp dụng

| Layer | Kỹ thuật | Tác dụng |
|-------|----------|---------|
| **Tiền xử lý âm thanh** | FFmpeg: Highpass + Lowpass + `afftdn` + `loudnorm` | Loại bỏ nhiễu nền, chuẩn hóa âm lượng |
| **Model** | `faster-whisper large-v3` (int8, CPU) | Chính xác nhất, hỗ trợ 99 ngôn ngữ |
| **Phát hiện ngôn ngữ** | Auto-detect → quyết định `language` param | Không bị lock sai ngôn ngữ |
| **Decoding** | `beam_size=10`, `temperature=0`, `best_of=5` | Kết quả ổn định, không hallucinate |
| **Lọc im lặng** | `vad_filter=True` (Silero VAD) | Cắt khoảng lặng, không sinh văn bản ảo |
| **Ngữ cảnh** | `initial_prompt` song ngữ Việt–Anh | Model nhận diện thuật ngữ chuyên ngành |
| **Hậu xử lý** | Lọc segment theo `no_speech_prob` | Chỉ giữ lại đoạn chứa giọng nói thật |

---

## 📋 Thứ tự chạy
1. `[Setup]` → `[Cấu hình]` → `[Phát hiện ngôn ngữ]` → `[Tiền xử lý]` → `[Bóc băng]` → `[Đánh giá]` → `[LLM Summary]`

---
## ⚙️ Cell 1 — Setup & Kiểm tra môi trường

In [ ]:
import os, sys, subprocess, shutil, warnings
warnings.filterwarnings('ignore')

# Thêm thư mục gốc vào Python path
ROOT_DIR = os.getcwd()
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

print(f"📁 Thư mục gốc: {ROOT_DIR}")

# Kiểm tra các dependency cần thiết
checks = {
    "faster-whisper": "faster_whisper",
    "ffmpeg": None,
    "requests": "requests",
}

print("\n🔍 Kiểm tra môi trường:")
all_ok = True

for name, module in checks.items():
    if module:  # Python package
        try:
            __import__(module)
            print(f"   ✅ {name}")
        except ImportError:
            print(f"   ❌ {name} — Cài bằng: pip install {name}")
            all_ok = False
    else:  # System binary
        if shutil.which("ffmpeg"):
            print(f"   ✅ {name}")
        else:
            print(f"   ❌ {name} — Cần cài FFmpeg và thêm vào PATH")
            all_ok = False

print()
if all_ok:
    print("🎉 Tất cả dependency sẵn sàng!")
else:
    print("⚠️  Một số dependency còn thiếu. Vui lòng cài đặt trước khi tiếp tục.")

---
## 🎛️ Cell 2 — Cấu hình bóc băng

> **Bạn chỉ cần chỉnh sửa ô này.** Mọi thứ còn lại sẽ tự động.

In [ ]:
# ============================================================
#  🎯 CẤU HÌNH CHÍNH — Chỉnh sửa các biến bên dưới
# ============================================================

# 1. Đường dẫn file audio cần bóc băng
#    Hỗ trợ: .mp3, .m4a, .wav, .mp4, .ogg, .flac, .webm
AUDIO_PATH = r"E:\TOM\Chap_6\QLDA\CursorProject\Antigravity_1\TestMeeting.m4a"

# 2. Ngôn ngữ ưu tiên
#    "vi"   = Tiếng Việt (phát biểu chủ yếu bằng tiếng Việt)
#    "en"   = Tiếng Anh  (phát biểu chủ yếu bằng tiếng Anh)
#    None   = Tự động phát hiện (dùng khi không chắc)
LANGUAGE_HINT = None  # Để None → sẽ auto-detect ở Cell 3

# 3. Model Whisper
#    "base"      ~74MB  — Nhanh, độ chính xác thấp
#    "medium"    ~1.5GB — Cân bằng tốt
#    "large-v3" ~3GB   — CHÍNH XÁC NHẤT ✅ (khuyến nghị)
WHISPER_MODEL = "large-v3"

# ============================================================
#  CÁC THAM SỐ NÂNG CAO (thường không cần đổi)
# ============================================================
BEAM_SIZE          = 10    # Rộng hơn → chính xác hơn (mặc định 5)
TEMPERATURE        = 0.0   # 0 = deterministic, không hallucinate
NO_SPEECH_THRESH   = 0.6   # Segment có >60% khả năng im lặng → bỏ qua
VAD_MIN_SILENCE_MS = 300   # Khoảng im lặng tối thiểu để cắt segment (ms)

# ============================================================

# Kiểm tra file tồn tại
if not os.path.exists(AUDIO_PATH):
    print(f"❌ Không tìm thấy file: {AUDIO_PATH}")
    print("💡 Hãy đổi AUDIO_PATH thành đường dẫn file audio thực tế của bạn.")
else:
    size_mb = os.path.getsize(AUDIO_PATH) / (1024 * 1024)
    ext = os.path.splitext(AUDIO_PATH)[1].upper()
    print(f"✅ File âm thanh hợp lệ:")
    print(f"   📂 Đường dẫn : {AUDIO_PATH}")
    print(f"   📦 Kích thước : {size_mb:.1f} MB")
    print(f"   🎵 Định dạng  : {ext}")
    print(f"\n⚙️  Cấu hình bóc băng:")
    print(f"   🤖 Model      : {WHISPER_MODEL}")
    print(f"   🌐 Ngôn ngữ   : {'Auto-detect (Cell 3)' if LANGUAGE_HINT is None else LANGUAGE_HINT}")
    print(f"   🔭 Beam size  : {BEAM_SIZE}")
    print(f"   🌡️  Temperature: {TEMPERATURE}")

---
## 🌐 Cell 3 — Phát hiện ngôn ngữ tự động

Bước này dùng Whisper để **nghe thử 30 giây đầu** và tự động xác định ngôn ngữ.
Nếu bạn đã biết ngôn ngữ, bỏ qua cell này và đặt `LANGUAGE_HINT = "vi"` hoặc `"en"` ở Cell 2.

In [ ]:
from faster_whisper import WhisperModel

# Nếu người dùng đã có LANGUAGE_HINT → bỏ qua detect
if LANGUAGE_HINT is not None:
    DETECTED_LANGUAGE = LANGUAGE_HINT
    print(f"✅ Ngôn ngữ đã được chỉ định thủ công: '{DETECTED_LANGUAGE}' — Bỏ qua auto-detect.")
else:
    print("🔍 Đang phát hiện ngôn ngữ từ file audio (nghe thử 30 giây đầu)...")
    print("⏳ Đang tải model tiny để detect nhanh...")

    # Dùng model "tiny" để detect ngôn ngữ — rất nhanh, không cần large-v3
    _tiny_model = WhisperModel("tiny", device="cpu", compute_type="int8")

    # Detect ngôn ngữ từ 30 giây đầu (không cần transcribe toàn bộ)
    _segments, _info = _tiny_model.transcribe(
        AUDIO_PATH,
        language=None,         # Không gán ngôn ngữ → auto detect
        beam_size=1,           # Nhanh nhất có thể
        vad_filter=True,
        max_new_tokens=1,      # Chỉ cần detect ngôn ngữ, không cần transcribe
    )
    # Consume iterator để lấy info
    for _ in _segments:
        break

    DETECTED_LANGUAGE = _info.language
    confidence = _info.language_probability

    # Hệ thống phân loại ưu tiên tiếng Việt & tiếng Anh
    SUPPORTED_PRIORITY = ["vi", "en"]
    if DETECTED_LANGUAGE not in SUPPORTED_PRIORITY:
        print(f"⚠️  Phát hiện ngôn ngữ '{DETECTED_LANGUAGE}' (độ tin cậy: {confidence:.1%})")
        print("⚠️  Ngôn ngữ này không phải Việt/Anh. Đề xuất đặt LANGUAGE_HINT = 'vi' hoặc 'en' thủ công.")
    else:
        lang_name = "Tiếng Việt 🇻🇳" if DETECTED_LANGUAGE == "vi" else "Tiếng Anh 🇬🇧"
        conf_bar = "█" * int(confidence * 20) + "░" * (20 - int(confidence * 20))
        print(f"\n✅ Kết quả phát hiện ngôn ngữ:")
        print(f"   🌐 Ngôn ngữ   : {lang_name} ('{DETECTED_LANGUAGE}')")
        print(f"   📊 Độ tin cậy : [{conf_bar}] {confidence:.1%}")
        if confidence < 0.7:
            print(f"   ⚠️  Độ tin cậy thấp! Hãy đặt LANGUAGE_HINT thủ công nếu kết quả không đúng.")

print(f"\n🎯 Ngôn ngữ sẽ dùng cho bóc băng: '{DETECTED_LANGUAGE}'")

---
## 🔊 Cell 4 — Tiền xử lý âm thanh (FFmpeg)

Áp dụng chuỗi bộ lọc chuyên nghiệp:
- **`highpass=f=100`** — Cắt tần số quá thấp (ồn phòng, gió, quạt)
- **`lowpass=f=8000`** — Giữ lại đúng dải tần giọng nói người (100–8000 Hz)
- **`afftdn=nf=-20`** — Khử nhiễu nền thích nghi (Adaptive FFT Denoiser)
- **`loudnorm`** — Chuẩn hóa âm lượng EBU R128 (tránh quá to/nhỏ)

In [ ]:
import uuid, time

UPLOADS_DIR = os.path.join(ROOT_DIR, "backend", "uploads")
os.makedirs(UPLOADS_DIR, exist_ok=True)

# Tạo tên file output
safe_name = os.path.splitext(os.path.basename(AUDIO_PATH))[0].replace(" ", "_")
PROCESSED_PATH = os.path.join(UPLOADS_DIR, f"test_{uuid.uuid4().hex[:6]}_{safe_name}.wav")

# Bộ lọc âm thanh chuẩn cho STT
AUDIO_FILTER = (
    "highpass=f=100,"
    "lowpass=f=8000,"
    "afftdn=nf=-20,"
    "loudnorm=I=-16:TP=-1.5:LRA=11"
)

cmd = [
    "ffmpeg", "-y",
    "-i", AUDIO_PATH,
    "-af", AUDIO_FILTER,
    "-ar", "16000",   # 16kHz — tần số chuẩn cho Whisper
    "-ac", "1",       # Mono
    "-c:a", "pcm_s16le",
    PROCESSED_PATH
]

print("🔧 Đang tiền xử lý âm thanh...")
print(f"   Bộ lọc : {AUDIO_FILTER}")
print(f"   Output  : {PROCESSED_PATH}")

t_start = time.time()
result = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.time() - t_start

if result.returncode != 0:
    print(f"\n❌ FFmpeg lỗi:\n{result.stderr}")
else:
    out_size = os.path.getsize(PROCESSED_PATH) / (1024 * 1024)
    print(f"\n✅ Tiền xử lý hoàn tất trong {elapsed:.1f}s")
    print(f"   📦 Kích thước file sau xử lý: {out_size:.1f} MB")
    print(f"   🎚️  Sample rate: 16000 Hz | Channels: Mono | Codec: PCM 16-bit")

---
## 🚀 Cell 5 — Bóc băng chính thức (Faster-Whisper large-v3)

Đây là bước bóc băng với toàn bộ tối ưu được áp dụng.

> ⚠️ **Lần đầu chạy:** Model `large-v3` (~3GB) sẽ tự động tải về cache. Các lần sau rất nhanh.

In [ ]:
from faster_whisper import WhisperModel
import time

# ---------------------------------------------------------------------------
# Initial Prompt song ngữ — giúp Whisper nhận diện thuật ngữ cuộc họp tốt hơn
# ---------------------------------------------------------------------------
INITIAL_PROMPTS = {
    "vi": (
        "Đây là bản ghi âm cuộc họp. Nội dung gồm: tiến độ dự án, phân công công việc, "
        "kết quả báo cáo, kế hoạch kinh doanh. "
        "Thuật ngữ: deadline, sprint, KPI, OKR, feedback, review, stakeholder."
    ),
    "en": (
        "This is a meeting recording. Topics include: project progress, task assignments, "
        "report results, and business planning. "
        "Terms: deadline, sprint, KPI, OKR, feedback, review, stakeholder."
    ),
}
# Nếu không phải vi/en, dùng prompt tiếng Anh làm mặc định
initial_prompt = INITIAL_PROMPTS.get(DETECTED_LANGUAGE, INITIAL_PROMPTS["en"])

# ---------------------------------------------------------------------------
# Tải model
# ---------------------------------------------------------------------------
print(f"🤖 Đang tải model Faster-Whisper '{WHISPER_MODEL}' (lần đầu ~3GB)...")
t0 = time.time()
# CPU + int8: giảm RAM 50%, tốc độ chấp nhận được
# Nếu có GPU NVIDIA: đổi device="cuda", compute_type="float16" → nhanh 10-15x
model = WhisperModel(WHISPER_MODEL, device="cpu", compute_type="int8")
print(f"✅ Model sẵn sàng ({time.time()-t0:.1f}s)")

# ---------------------------------------------------------------------------
# Bóc băng với toàn bộ tối ưu
# ---------------------------------------------------------------------------
print(f"\n🎙️  Bắt đầu bóc băng...")
print(f"   Ngôn ngữ : '{DETECTED_LANGUAGE}' | Beam: {BEAM_SIZE} | Temp: {TEMPERATURE} | VAD: ON")
t_start = time.time()

segments, info = model.transcribe(
    PROCESSED_PATH,

    # --- Cấu hình ngôn ngữ ---
    language=DETECTED_LANGUAGE,        # Khai báo tường minh, không để auto-detect sai
    initial_prompt=initial_prompt,     # Ngữ cảnh cuộc họp theo đúng ngôn ngữ

    # --- Cấu hình decoding: ưu tiên chính xác ---
    beam_size=BEAM_SIZE,               # 10 > 5 (mặc định): tìm kiếm rộng hơn
    best_of=5,                         # Lấy best trong 5 candidate
    temperature=TEMPERATURE,           # 0.0 = deterministic, không hallucinate

    # --- Voice Activity Detection ---
    vad_filter=True,                   # Silero VAD: lọc khoảng im lặng
    vad_parameters=dict(
        min_silence_duration_ms=VAD_MIN_SILENCE_MS,
        speech_pad_ms=200,             # Biên đệm để không cắt đứt giữa câu
    ),

    # --- Cấu hình chất lượng ---
    condition_on_previous_text=True,   # Dùng ngữ cảnh đoạn trước → câu liền mạch
    word_timestamps=True,              # Timestamps từng chữ (hữu ích cho debug)
    no_speech_threshold=NO_SPEECH_THRESH,  # Ngưỡng bỏ segment nhiễu
    log_prob_threshold=-1.0,           # Giữ ngưỡng rộng để không bỏ sót
    compression_ratio_threshold=2.4,   # Bỏ segment bị lặp liên tục (hallucination)
)

# ---------------------------------------------------------------------------
# Thu thập kết quả từ generator
# ---------------------------------------------------------------------------
valid_segments = []
skipped_count  = 0
all_segment_data = []

for seg in segments:
    seg_info = {
        "start": seg.start,
        "end":   seg.end,
        "text":  seg.text.strip(),
        "no_speech_prob": seg.no_speech_prob,
        "avg_logprob":    seg.avg_logprob,
    }
    all_segment_data.append(seg_info)

    if seg.no_speech_prob > NO_SPEECH_THRESH:
        skipped_count += 1
        continue
    valid_segments.append(seg.text.strip())

elapsed = time.time() - t_start

# Ghép transcript
TRANSCRIPT = " ".join(valid_segments)

# ---------------------------------------------------------------------------
# Kết quả
# ---------------------------------------------------------------------------
print(f"\n{'='*65}")
print(f"📋 KẾT QUẢ BÓC BĂNG")
print(f"{'='*65}")
print(TRANSCRIPT)
print(f"{'='*65}")
print(f"\n📊 Thống kê:")
print(f"   🌐 Ngôn ngữ phát hiện : '{info.language}' (tin cậy: {info.language_probability:.1%})")
print(f"   📝 Số từ              : {len(TRANSCRIPT.split())}")
print(f"   🔤 Số ký tự           : {len(TRANSCRIPT)}")
print(f"   📦 Số segment hợp lệ  : {len(valid_segments)}/{len(all_segment_data)}")
print(f"   🔕 Segment bị bỏ qua  : {skipped_count} (nhiễu/im lặng)")
print(f"   ⏱️  Thời gian xử lý    : {elapsed:.1f}s")

---
## 🔬 Cell 6 — Đánh giá chất lượng chi tiết (Segments)

Xem từng đoạn để kiểm tra timestamp và xác suất im lặng — hữu ích để debug.

In [ ]:
if not all_segment_data:
    print("⚠️  Chưa có dữ liệu segment. Hãy chạy Cell 5 trước.")
else:
    print(f"📑 CHI TIẾT TỪNG SEGMENT ({len(all_segment_data)} tổng cộng)")
    print(f"{'─'*75}")
    print(f"{'#':<4} {'Thời gian':<18} {'No-speech':<11} {'Trạng thái':<10} Nội dung")
    print(f"{'─'*75}")

    for i, seg in enumerate(all_segment_data, 1):
        time_str  = f"{seg['start']:6.1f}s → {seg['end']:6.1f}s"
        ns_prob   = seg['no_speech_prob']
        status    = "🔕 SKIP" if ns_prob > NO_SPEECH_THRESH else "✅ OK  "
        ns_bar    = f"{ns_prob:.2f}"
        text_preview = seg['text'][:60] + ("..." if len(seg['text']) > 60 else "")
        print(f"{i:<4} {time_str:<18} {ns_bar:<11} {status:<10} {text_preview}")

    print(f"{'─'*75}")

    # Phân tích chất lượng
    ns_probs = [s['no_speech_prob'] for s in all_segment_data]
    avg_ns   = sum(ns_probs) / len(ns_probs) if ns_probs else 0
    valid_r  = len(valid_segments) / len(all_segment_data) * 100 if all_segment_data else 0

    print(f"\n📊 Đánh giá chất lượng audio:")
    print(f"   Tỷ lệ segment hợp lệ  : {valid_r:.0f}%")
    print(f"   No-speech prob trung bình: {avg_ns:.3f}")

    if valid_r >= 80:
        print("   🟢 Chất lượng audio: TỐT")
    elif valid_r >= 60:
        print("   🟡 Chất lượng audio: TRUNG BÌNH — Nên thử file audio rõ hơn")
    else:
        print("   🔴 Chất lượng audio: KÉM — Audio nhiều nhiễu, kết quả có thể không chuẩn")

---
## 🤖 Cell 7 — Phân tích & Tóm tắt bằng LLM (Ollama)

Gửi transcript vừa bóc sang Llama 3.2 để tóm tắt và trích xuất action items.

In [ ]:
import requests as http_requests

# Kiểm tra Ollama
try:
    ping = http_requests.get("http://localhost:11434/api/tags", timeout=5)
    models_available = [m['name'] for m in ping.json().get('models', [])]
    print(f"✅ Ollama đang chạy | Các model sẵn có: {models_available}")
except Exception:
    print("❌ Không kết nối được Ollama! Hãy chạy 'ollama serve' ở terminal khác.")
    raise SystemExit()

In [ ]:
from backend.services.llm_service import generate_meeting_summary
import json

if 'TRANSCRIPT' not in dir() or not TRANSCRIPT:
    print("⚠️  Chưa có transcript. Hãy chạy Cell 5 trước.")
else:
    print(f"🚀 Đang gửi {len(TRANSCRIPT.split())} từ sang Ollama để phân tích...\n")
    try:
        t_llm = time.time()
        summary_result = generate_meeting_summary(TRANSCRIPT)
        llm_elapsed = time.time() - t_llm

        print(f"{'='*65}")
        print(f"🧠 KẾT QUẢ PHÂN TÍCH AI (hoàn tất trong {llm_elapsed:.1f}s)")
        print(f"{'='*65}")
        print(json.dumps(summary_result, indent=4, ensure_ascii=False))
        print(f"{'='*65}")

        # Xuất transcript ra file .txt
        output_txt = os.path.join(ROOT_DIR, "transcript_output.txt")
        with open(output_txt, "w", encoding="utf-8") as f:
            f.write("=== TRANSCRIPT ===\n")
            f.write(TRANSCRIPT)
            f.write("\n\n=== AI SUMMARY ===\n")
            f.write(json.dumps(summary_result, indent=4, ensure_ascii=False))
        print(f"\n💾 Đã lưu kết quả ra: {output_txt}")

    except Exception as e:
        import traceback
        print(f"❌ Lỗi LLM: {e}")
        traceback.print_exc()